# 固定済み before / after ペアの Lab 解析

この notebook は探索をしません。`makeup.mp4` なら `outputs/makeup/selected_pair.json` を使います。
探索側と同じ `VIDEO` ファイル名を指定するだけです。


In [ ]:
from pathlib import Path
import os

cwd = Path.cwd().resolve()
marker = Path('analysis/analyze_cheek_lab.py')
if (cwd / marker).is_file():
    REPO_ROOT = cwd
elif cwd.name == 'notebooks' and (cwd.parent / marker).is_file():
    REPO_ROOT = cwd.parent
    os.chdir(REPO_ROOT)
else:
    raise RuntimeError(f'ikiikimake のリポジトリ直下または notebooks/ から実行してください: current={cwd}')

print('repo root:', Path.cwd())

## 設定


In [ ]:
VIDEO = Path('makeup.mp4')
OUTPUT = Path('outputs') / VIDEO.stem
SELECTED = OUTPUT / 'selected_pair.json'
LAB_OUTPUT = OUTPUT / 'lab_selected_pair'

if not SELECTED.is_file():
    raise FileNotFoundError(
        f'承認済みペアがありません: {SELECTED}. '
        '先に video_before_after_pair.ipynb で候補を固定してください。'
    )

print('selected pair:', SELECTED)
print('lab output   :', LAB_OUTPUT)

## 固定済みペアを検証して解析

元画像と ROI 成果物の SHA-256 を再検証します。既存の Lab 出力が同じ入力から作られたものなら再利用し、`summary.html` がなければ追加します。入力が違えば停止します。


In [ ]:
import hashlib
import json

from analysis.analyze_cheek_lab import analyze_pair, write_summary_html

if not SELECTED.is_file():
    raise FileNotFoundError(f'selected_pair.json がありません: {SELECTED}')

selected = json.loads(SELECTED.read_text(encoding='utf-8'))
if Path(selected.get('video_path', '')).resolve() != VIDEO.resolve():
    raise RuntimeError('selected_pair.json の動画と VIDEO が一致しません。')

def sha256_file(path: Path) -> str:
    if not path.is_file():
        raise FileNotFoundError(path)
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

def checked_file(path: Path, expected_sha256: str, label: str) -> Path:
    actual = sha256_file(path)
    if actual != expected_sha256:
        raise RuntimeError(
            f'{label} の SHA-256 が一致しません。'
            f' expected={expected_sha256} actual={actual} path={path}'
        )
    return path

resolved = {}
for phase in ('before', 'after'):
    item = selected.get(phase)
    if not isinstance(item, dict):
        raise ValueError(f'selected_pair.json に {phase} がありません。')
    image = checked_file(Path(item['image_path']), item['image_sha256'], f'{phase} image')
    roi_dir = Path(item['roi_dir'])
    masks = checked_file(roi_dir / 'roi_masks.npz', item['roi_masks_sha256'], f'{phase} roi_masks')
    checked_file(roi_dir / 'roi_points.json', item['roi_points_sha256'], f'{phase} roi_points')
    checked_file(roi_dir / 'roi_overlay.png', item['roi_overlay_sha256'], f'{phase} roi_overlay')
    resolved[phase] = {'image': image, 'masks': masks}

summary_path = LAB_OUTPUT / 'analysis_summary.json'
if LAB_OUTPUT.exists():
    if not summary_path.is_file():
        raise FileExistsError(
            f'Lab出力フォルダはありますが analysis_summary.json がありません: {LAB_OUTPUT}'
        )
    summary = json.loads(summary_path.read_text(encoding='utf-8'))
    expected_inputs = {
        'before': str(resolved['before']['image'].resolve()),
        'after': str(resolved['after']['image'].resolve()),
        'before_masks': str(resolved['before']['masks'].resolve()),
        'after_masks': str(resolved['after']['masks'].resolve()),
    }
    if summary.get('inputs') != expected_inputs:
        raise RuntimeError(
            '既存Lab解析の入力が現在の selected_pair.json と一致しません。'
            f' expected={expected_inputs} saved={summary.get("inputs")}'
        )
    summary_html = write_summary_html(LAB_OUTPUT, summary)
    print('既存Lab解析を検証して再利用しました:', LAB_OUTPUT)
    print('summary:', summary_html)
else:
    summary = analyze_pair(
        resolved['before']['image'],
        resolved['after']['image'],
        resolved['before']['masks'],
        resolved['after']['masks'],
        LAB_OUTPUT,
    )
    print('Lab analysis complete:', LAB_OUTPUT)
    print('summary:', LAB_OUTPUT / 'summary.html')

## 結果

`delta` は after − before、`delta_minus_forehead` は `(頬 after − before) − (額 after − before)` です。額差し引きは未検証の control 比較で、照明補正やメイク効果の確定値ではありません。


In [ ]:
from IPython.display import HTML, display

summary_path = LAB_OUTPUT / 'analysis_summary.json'
if not summary_path.is_file():
    raise FileNotFoundError(summary_path)
summary = json.loads(summary_path.read_text(encoding='utf-8'))

rows = [row for row in summary['deltas'] if row['metric'] in ('a_median', 'a_mean')]
if not rows:
    raise RuntimeError('a* の差分結果がありません。')

headers = ('side', 'metric', 'before', 'after', 'delta', 'forehead_delta', 'delta_minus_forehead')
head = ''.join(f'<th>{name}</th>' for name in headers)
body = []
for row in rows:
    cells = []
    for name in headers:
        value = row[name]
        cells.append(f'<td>{value:.3f}</td>' if isinstance(value, float) else f'<td>{value}</td>')
    body.append('<tr>' + ''.join(cells) + '</tr>')

display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + ''.join(body) + '</tbody></table>'))

## 生成ファイル


In [ ]:
required = [
    'summary.html',
    'analysis_summary.json',
    'lab_stats.csv',
    'lab_deltas.csv',
    'roi_samples.png',
    'left_cheek_lab_hist.png',
    'right_cheek_lab_hist.png',
    'forehead_lab_hist.png',
]
for name in required:
    path = LAB_OUTPUT / name
    if not path.is_file():
        raise FileNotFoundError(path)
    print(path)

print('\nまず見るファイル:', (LAB_OUTPUT / 'summary.html').resolve())